### 1. Data Initialization

In [1]:
"""
classical_models_validation.py
================================
Validação dos modelos ETS, ARIMA/SARIMA e SARIMAX para previsão semanal
de casos de dengue por regional_geocode — Desafio IMDC 2026.

Adaptado do script R (ETS + ARIMA/SARIMA) com:
  • Dados semanais (frequência = 52)
  • 4 janelas de validação definidas pelas flags train_*/target_*
  • Intervalo de previsão: mediana + 50%, 80%, 90%, 95%
  • Métricas: MAE, RMSE, MAPE, WIS
  • SARIMAX: mesmas etapas do ARIMA + variáveis exógenas
  • ARIMA : auto_arima com m=52, busca sazonal restrita (P,Q ≤ 1) — SARIMA real
  • ETS   : conversão explícita para float64 puro antes do fit; fallback para ANN
  • SARIMAX: seasonal_order removido (Fourier já cobre sazonalidade; period=1 era inválido)
  • Plots : apenas salvos em disco (sem plt.show), relatório TXT por região
  • Saída : gráficos por UF (soma de regiões) e heatmap WIS modelo × UF por validation
"""

# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
 
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import kpss
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.tsa.seasonal import STL
 
import pmdarima as pm
 
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Configurações globais
# ---------------------------------------------------------------------------
BASE_DIR = Path(r"C:\myenv\env\Mestrado\2026_imdc_challenge")

DB_DIR  = BASE_DIR / "0_databases"

OUT_DIR    = BASE_DIR / "2_1_results_classical"
OUT_DIR.mkdir(exist_ok=True)

In [2]:
FREQ      = 52
K_FOURIER = 4
TARGET    = "casos"
 
DROP_EXOG = {
    "pop_density_km2","BSh_koppen","As_koppen","rel_humid_med",
    "casos","date","year","epiweek","uf","uf_code",
    "regional_geocode","regional_health_name","macroregion_code","macroregion_name",
    "train_1","train_2","train_3","train_4",
    "target_1","target_2","target_3","target_4",
    "regional_health_area_km2",
}
 
VALIDATIONS = [
    (1, "train_1", "target_1"),
    (2, "train_2", "target_2"),
    (3, "train_3", "target_3"),
    (4, "train_4", "target_4"),
]
 
PI_LEVELS = [0.50, 0.80, 0.90, 0.95]

### 2. Loading Functions

In [3]:
# ---------------------------------------------------------------------------
# 0. Carga e agregação para UF
# ---------------------------------------------------------------------------
print("Carregando hierarch_forecast.parquet...")
hf_raw = pd.read_parquet(DB_DIR / "hierarch_forecast.parquet")
hf_raw = hf_raw[hf_raw["uf"] != "ES"]
hf_raw["date"]  = pd.to_datetime(hf_raw["date"])
hf_raw["casos"] = hf_raw["casos"].astype(float)
hf_raw["enso"]  = hf_raw["enso"].ffill().bfill()
 
# Colunas dummies (serão somadas e clipadas em 1)
koppen_cols = [c for c in hf_raw.columns if c.endswith("_koppen")]
biome_cols  = [c for c in hf_raw.columns if c.endswith("_biome")]
dummy_cols  = koppen_cols + biome_cols
 
# Colunas climáticas contínuas (média)
climate_cols = ["temp_med","precip_med","pressure_med",
                "thermal_range","rainy_days","enso"]
 
# Flags booleanas — mesma data tem o mesmo valor em todas as regiões da UF
flag_cols = ["train_1","train_2","train_3","train_4",
             "target_1","target_2","target_3","target_4"]
 
agg_dict = {
    "casos"     : "sum",
    "population": "sum",
    "uf_code"   : "first",
    "macroregion_code": "first",
    "macroregion_name": "first",
    "year"      : "first",
    "epiweek"   : "first",
    **{c: "mean" for c in climate_cols},
    **{c: "sum"  for c in dummy_cols},
    **{c: "first" for c in flag_cols},
}
 
print("  Agregando para nível UF...")
hf = (hf_raw.groupby(["uf","date"], sort=True)
      .agg(agg_dict)
      .reset_index())
 
# Dummies: clip em 1 (presença/ausência no nível UF)
hf[dummy_cols] = hf[dummy_cols].clip(upper=1)
 
# Flags de volta para bool
for c in flag_cols:
    hf[c] = hf[c].astype(bool)
 
hf = hf.sort_values(["uf","date"]).reset_index(drop=True)
 
# Exógenas disponíveis após agregação
exog_cols = sorted(
    set(hf.columns) - DROP_EXOG
    - {c for c in hf.columns if hf[c].dtype == object}
)
print(f"  Exógenas SARIMAX ({len(exog_cols)}): {exog_cols}")
 
# Mapeamento UF → uf_code
uf_code_map = (hf[["uf","uf_code"]].drop_duplicates("uf").set_index("uf"))
 
ufs = sorted(hf["uf"].unique())
print(f"  UFs: {len(ufs)}  →  {ufs}")
print(f"  AVISO: {len(ufs)} UFs × 4 validações × 3 modelos\n")


# ===========================================================================
# FUNÇÕES AUXILIARES
# ===========================================================================

def wis(y_true, median, lower, upper, levels):
    K = len(levels)
    s = 0.5 * np.abs(y_true - median)
    for a in levels:
        IS = (upper[a] - lower[a]
              + (2/a) * np.maximum(0, lower[a] - y_true)
              + (2/a) * np.maximum(0, y_true - upper[a]))
        s += (a/2) * IS
    return float(np.mean(s) / (K + 0.5))


def error_metrics(y_true, y_pred, label, median=None, lower=None, upper=None, levels=None):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mask = ~(np.isnan(yt) | np.isnan(yp))
    yt, yp = yt[mask], yp[mask]
    mae  = float(np.mean(np.abs(yt - yp)))
    rmse = float(np.sqrt(np.mean((yt - yp)**2)))
    denom = np.where(yt == 0, np.nan, yt)
    mape = float(np.nanmean(np.abs((yt - yp) / denom)) * 100)
    row  = {"model": label, "MAE": mae, "RMSE": rmse, "MAPE": mape}
    if median is not None:
        row["WIS"] = wis(yt, np.array(median)[mask],
                         {a: np.array(lower[a])[mask] for a in levels},
                         {a: np.array(upper[a])[mask] for a in levels}, levels)
    return row


def breusch_pagan_pval(y):
    x = np.hstack([np.ones((len(y),1)), np.arange(len(y)).reshape(-1,1)])
    res = OLS(y, x).fit()
    _, p, _, _ = het_breuschpagan(res.resid, x)
    return p


def n_diffs_kpss(y, max_d=2):
    for d in range(max_d+1):
        s = np.diff(y, n=d) if d > 0 else y
        try:
            _, p, _, _ = kpss(s, regression="c", nlags="auto")
            if p >= 0.05: return d
        except: return d
    return max_d


def fourier_terms(start, n, period=FREQ, K=K_FOURIER):
    t = np.arange(start, start + n)
    return np.column_stack([
        f(2*np.pi*k*t/period)
        for k in range(1, K+1) for f in (np.sin, np.cos)
    ])


def limpar(arr):
    df = pd.DataFrame(arr, dtype=float)
    df = df.replace([np.inf,-np.inf], np.nan).ffill().bfill().fillna(0)
    return df.values


def ets_intervals(fit, h, levels=PI_LEVELS, n_sim=500):
    try:
        sim = fit.simulate(nsimulations=h, repetitions=n_sim, error="add")
        lo = {a: np.quantile(sim,(1-a)/2,    axis=1) for a in levels}
        hi = {a: np.quantile(sim,1-(1-a)/2,  axis=1) for a in levels}
    except Exception:
        fc    = fit.forecast(h)
        sigma = np.std(fit.resid) * np.sqrt(np.arange(1, h+1)) + 1e-6
        from scipy.stats import norm
        lo = {a: fc + norm.ppf((1-a)/2)    * sigma for a in levels}
        hi = {a: fc + norm.ppf(1-(1-a)/2)  * sigma for a in levels}
    return lo, hi


def save_forecast_plot(train_dates, train_vals, target_dates, target_vals,
                       fc_dates, fc_med, fc_lo, fc_hi, title, path):
    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(train_dates,  train_vals,  color="#2166ac", lw=0.7, label="Treino")
    ax.plot(target_dates, target_vals, color="black",   lw=1.2, label="Observado")
    pal = ["#f4a582","#d6604d","#b2182b","#67001f"]
    for i, a in enumerate(sorted(fc_lo.keys(), reverse=True)):
        ax.fill_between(fc_dates, fc_lo[a], fc_hi[a],
                        alpha=0.25, color=pal[i], label=f"IP {int(a*100)}%")
    ax.plot(fc_dates, fc_med, color="#d6604d", lw=1.2, ls="--", label="Mediana")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.set_ylabel("Casos"); ax.set_title(title, fontsize=9, fontweight="bold")
    ax.legend(fontsize=7, ncol=4); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def save_resid_plot(resid, title, path):
    resid = pd.Series(resid, dtype=float).dropna()
    fig, axes = plt.subplots(1,3, figsize=(13,3))
    axes[0].plot(resid.values, lw=0.6, color="#2166ac")
    axes[0].axhline(0, color="k", lw=0.8, ls="--"); axes[0].set_title("Resíduos")
    pd.plotting.autocorrelation_plot(resid, ax=axes[1])
    axes[1].set_xlim(0,52); axes[1].set_title("ACF Resíduos")
    axes[2].hist(resid, bins=30, color="#4393c3", edgecolor="white")
    axes[2].set_title("Histograma")
    fig.suptitle(title, fontsize=9, fontweight="bold"); plt.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight"); plt.close(fig)

Carregando hierarch_forecast.parquet...
  Agregando para nível UF...
  Exógenas SARIMAX (20): ['Af_koppen', 'Am_koppen', 'Amazônia_biome', 'Aw_koppen', 'Caatinga_biome', 'Cerrado_biome', 'Cfa_koppen', 'Cfb_koppen', 'Cwa_koppen', 'Cwb_koppen', 'Mata Atlântica_biome', 'Pampa_biome', 'Pantanal_biome', 'enso', 'population', 'precip_med', 'pressure_med', 'rainy_days', 'temp_med', 'thermal_range']
  UFs: 26  →  ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP', 'TO']
  AVISO: 26 UFs × 4 validações × 3 modelos



### 3. Forecast Pipeline

In [4]:
# ===========================================================================
# PIPELINE PRINCIPAL  — loop por UF
# ===========================================================================
all_forecasts, all_metrics = [], []
txt_lines = []
 
t0_total = time.time()
 
for i_uf, uf_label in enumerate(ufs):
 
    rd       = hf[hf["uf"] == uf_label].sort_values("date").reset_index(drop=True)
    uf_code  = int(uf_code_map.loc[uf_label, "uf_code"])
 
    txt_lines.append(f"\n{'='*70}")
    txt_lines.append(f"UF: {uf_label}  (uf_code={uf_code})")
    txt_lines.append(f"{'='*70}")
    print(f"[{i_uf+1}/{len(ufs)}] UF: {uf_label}")
 
    # Análise preliminar
    y_full  = rd[TARGET].values.astype(float)
    bp_pval = breusch_pagan_pval(np.clip(y_full, 1e-3, None))
    use_log = bp_pval < 0.05
    y_tr    = np.log1p(y_full) if use_log else y_full
    d_ord   = n_diffs_kpss(y_tr)
    inv     = (lambda x: np.expm1(x)) if use_log else (lambda x: x)
 
    txt_lines.append(f"Breusch-Pagan p={bp_pval:.4f} → {'log1p' if use_log else 'nível'}")
    txt_lines.append(f"KPSS d sugerido = {d_ord}")
 
    # STL
    try:
        stl = STL(y_tr, period=FREQ, robust=True).fit()
        fig_stl, ax_stl = plt.subplots(4,1, figsize=(13,8), sharex=True)
        for ax_s, comp, nm in zip(ax_stl,
            [y_tr, stl.trend, stl.seasonal, stl.resid],
            ["Série","Tendência","Sazonalidade","Resíduo"]):
            ax_s.plot(rd["date"].values, comp, lw=0.7, color="#2166ac")
            ax_s.set_ylabel(nm, fontsize=8)
            ax_s.spines[["top","right"]].set_visible(False)
        ax_stl[0].set_title(f"STL — {uf_label}", fontsize=9, fontweight="bold")
        plt.tight_layout()
        fig_stl.savefig(OUT_DIR / f"{uf_label}_stl.png", dpi=120, bbox_inches="tight")
        plt.close(fig_stl)
    except Exception as e:
        txt_lines.append(f"[AVISO] STL: {e}")
 
    # ------------------------------------------------------------------
    for val_num, train_flag, target_flag in VALIDATIONS:
 
        txt_lines.append(f"\n  -- Validation {val_num} --")
        train_mask  = rd[train_flag].astype(bool)
        target_mask = rd[target_flag].astype(bool)
 
        if train_mask.sum() < FREQ or target_mask.sum() == 0:
            txt_lines.append("  [SKIP]")
            continue
 
        tr  = rd[train_mask].sort_values("date")
        tgt = rd[target_mask].sort_values("date")
 
        y_tr_raw  = tr[TARGET].values.astype(float)
        y_tgt_raw = tgt[TARGET].values.astype(float)
        y_train   = np.log1p(y_tr_raw)  if use_log else y_tr_raw
 
        last_train = tr["date"].max()
        first_tgt  = tgt["date"].min()
        gap        = int((first_tgt - last_train).days / 7)
        h_total    = gap + len(tgt)
 
        txt_lines.append(f"  Treino : {tr['date'].min().date()} → {last_train.date()} ({len(tr)} sem)")
        txt_lines.append(f"  Gap    : {gap} semanas")
        txt_lines.append(f"  Target : {first_tgt.date()} → {tgt['date'].max().date()} ({len(tgt)} sem)")
 
        fc_dates = pd.date_range(last_train + pd.Timedelta(weeks=1),
                                 periods=h_total, freq="W-SUN")
 
        # Exógenas
        exog_tr   = limpar(tr[exog_cols].astype(float).values)
        exog_hist = (rd.assign(w=rd["date"].dt.isocalendar().week.astype(int))
                       .groupby("w")[exog_cols].mean())
        fc_df_idx = rd[rd["date"].isin(fc_dates)].set_index("date")
        fc_rows   = []
        for d in fc_dates:
            if d in fc_df_idx.index:
                fc_rows.append(fc_df_idx.loc[d, exog_cols].values.astype(float))
            else:
                wk = min(int(d.isocalendar().week), 52)
                fc_rows.append(exog_hist.loc[wk].values if wk in exog_hist.index
                               else exog_hist.mean().values)
        exog_fc = limpar(np.array(fc_rows, dtype=float))
 
        n_tr    = len(y_train)
        four_tr = fourier_terms(0,    n_tr,    FREQ, K_FOURIER)
        four_fc = fourier_terms(n_tr, h_total, FREQ, K_FOURIER)
 
        sarimax_tr_exog = np.hstack([exog_tr, four_tr])
        sarimax_fc_exog = np.hstack([exog_fc, four_fc])
 
        results_val  = {}
        uf_out_dir   = OUT_DIR / uf_label
        uf_out_dir.mkdir(exist_ok=True)
 
        # ==============================================================
        # A. ARIMA / SARIMA  (parâmetros mantidos conforme solicitado)
        # ==============================================================
        print(f"    V{val_num} ARIMA...", end=" ", flush=True)
        t0 = time.time()
        try:
            arima_fit = pm.auto_arima(
                y_train,
                exogenous=four_tr,
                m=FREQ,
                seasonal=True,
                d=d_ord, D=None,
                max_p=2, max_q=2,
                max_P=1, max_Q=1,
                max_order=6,
                stepwise=True,
                approximation=True,
                information_criterion="aicc",
                error_action="ignore",
                suppress_warnings=True,
            )
            print(f"SARIMA{arima_fit.order}x{arima_fit.seasonal_order} "
                  f"({time.time()-t0:.0f}s)", flush=True)
            txt_lines.append(f"  ARIMA: SARIMA{arima_fit.order}x{arima_fit.seasonal_order} "
                             f"AICc={arima_fit.aic():.1f}")
 
            save_resid_plot(arima_fit.resid(),
                            f"Resíduos ARIMA V{val_num} | {uf_label}",
                            uf_out_dir / f"v{val_num}_arima_resid.png")
 
            ar_lo, ar_hi, ar_med = {}, {}, None
            for a in PI_LEVELS:
                fc_v, ci = arima_fit.predict(n_periods=h_total, exogenous=four_fc,
                                              return_conf_int=True, alpha=1-a)
                if ar_med is None: ar_med = fc_v
                ar_lo[a] = ci[:,0]; ar_hi[a] = ci[:,1]
 
            ar_med_i = inv(ar_med)
            ar_lo_i  = {a: inv(ar_lo[a]) for a in PI_LEVELS}
            ar_hi_i  = {a: inv(ar_hi[a]) for a in PI_LEVELS}
            tgt_med  = ar_med_i[gap:]
            m_ar = error_metrics(y_tgt_raw, tgt_med, "ARIMA", tgt_med,
                                 {a: ar_lo_i[a][gap:] for a in PI_LEVELS},
                                 {a: ar_hi_i[a][gap:] for a in PI_LEVELS}, PI_LEVELS)
            results_val["ARIMA"] = {"median":ar_med_i,"lower":ar_lo_i,
                                    "upper":ar_hi_i,"metrics":m_ar}
            txt_lines.append(f"    MAE={m_ar['MAE']:.1f}  RMSE={m_ar['RMSE']:.1f}  "
                             f"MAPE={m_ar['MAPE']:.1f}%  WIS={m_ar['WIS']:.2f}")
            save_forecast_plot(tr["date"].values, y_tr_raw,
                               tgt["date"].values, y_tgt_raw,
                               fc_dates, ar_med_i, ar_lo_i, ar_hi_i,
                               f"ARIMA V{val_num} | {uf_label}",
                               uf_out_dir / f"v{val_num}_arima_forecast.png")
        except Exception as e:
            print(f"ERRO ({time.time()-t0:.0f}s)", flush=True)
            txt_lines.append(f"  [ERRO] ARIMA: {e}")
 
        # ==============================================================
        # B. ETS  — três níveis de fallback para garantir convergência
        # ==============================================================
        print(f"    V{val_num} ETS...", end=" ", flush=True)
        t0 = time.time()
        try:
            # Série como ndarray float64 puro + offset mínimo contra zeros
            y_ets = np.array(y_train, dtype=np.float64)
            y_ets = np.where(np.isnan(y_ets), 0.0, y_ets) + 1e-2
 
            best_ets, best_aicc = None, np.inf
 
            candidates = [
                (None,  False, None ),   # ANN
                ("add", False, None ),   # AAN
                ("add", True,  None ),   # AAdN
                (None,  False, "add"),   # ANS
                ("add", False, "add"),   # AAS
                ("add", True,  "add"),   # AAdS
            ]
 
            # Nível 1: heuristic + optimized=True
            for (trend, damped, seasonal) in candidates:
                try:
                    m = ExponentialSmoothing(
                        y_ets,
                        trend=trend,
                        damped_trend=damped if trend else False,
                        seasonal=seasonal,
                        seasonal_periods=FREQ if seasonal else None,
                        initialization_method="heuristic",
                    ).fit(optimized=True, disp=False)
                    if m.aicc < best_aicc:
                        best_aicc, best_ets = m.aicc, m
                except Exception:
                    pass
 
            # Nível 2: heuristic + optimized=False (parâmetros fixos razoáveis)
            if best_ets is None:
                for (trend, damped, seasonal) in candidates:
                    try:
                        fit_kw = {"smoothing_level": 0.3, "optimized": False, "disp": False}
                        if trend:   fit_kw["smoothing_trend"]    = 0.1
                        if seasonal: fit_kw["smoothing_seasonal"] = 0.1
                        if damped:  fit_kw["damping_trend"]       = 0.9
                        m = ExponentialSmoothing(
                            y_ets,
                            trend=trend,
                            damped_trend=damped if trend else False,
                            seasonal=seasonal,
                            seasonal_periods=FREQ if seasonal else None,
                            initialization_method="heuristic",
                        ).fit(**fit_kw)
                        if best_ets is None:   # pega o primeiro que funcionar
                            best_ets = m
                            best_aicc = getattr(m, "aicc", np.inf)
                    except Exception:
                        pass
 
            # Nível 3: SimpleExpSmoothing — nunca falha
            if best_ets is None:
                best_ets = SimpleExpSmoothing(
                    y_ets, initialization_method="heuristic"
                ).fit(smoothing_level=0.3, optimized=False)
 
            cfg = (f"trend={getattr(best_ets.model,'trend',None)}  "
                   f"seasonal={getattr(best_ets.model,'seasonal',None)}  "
                   f"damped={getattr(best_ets.model,'damped_trend',False)}")
            print(f"{cfg} ({time.time()-t0:.0f}s)", flush=True)
            txt_lines.append(f"  ETS: {cfg}  AICc={getattr(best_ets,'aicc',float('nan')):.1f}")
 
            save_resid_plot(best_ets.resid,
                            f"Resíduos ETS V{val_num} | {uf_label}",
                            uf_out_dir / f"v{val_num}_ets_resid.png")
 
            fc_ets   = np.array(best_ets.forecast(h_total), dtype=float)
            ets_lo, ets_hi = ets_intervals(best_ets, h_total)
            fc_ets_i = inv(fc_ets)
            ets_lo_i = {a: inv(ets_lo[a]) for a in PI_LEVELS}
            ets_hi_i = {a: inv(ets_hi[a]) for a in PI_LEVELS}
            tgt_ets  = fc_ets_i[gap:]
            m_ets = error_metrics(y_tgt_raw, tgt_ets, "ETS", tgt_ets,
                                  {a: ets_lo_i[a][gap:] for a in PI_LEVELS},
                                  {a: ets_hi_i[a][gap:] for a in PI_LEVELS}, PI_LEVELS)
            results_val["ETS"] = {"median":fc_ets_i,"lower":ets_lo_i,
                                  "upper":ets_hi_i,"metrics":m_ets}
            txt_lines.append(f"    MAE={m_ets['MAE']:.1f}  RMSE={m_ets['RMSE']:.1f}  "
                             f"MAPE={m_ets['MAPE']:.1f}%  WIS={m_ets['WIS']:.2f}")
            save_forecast_plot(tr["date"].values, y_tr_raw,
                               tgt["date"].values, y_tgt_raw,
                               fc_dates, fc_ets_i, ets_lo_i, ets_hi_i,
                               f"ETS V{val_num} | {uf_label}",
                               uf_out_dir / f"v{val_num}_ets_forecast.png")
        except Exception as e:
            print(f"ERRO ({time.time()-t0:.0f}s)", flush=True)
            txt_lines.append(f"  [ERRO] ETS: {e}")
 
        # ==============================================================
        # C. SARIMAX
        # ==============================================================
        print(f"    V{val_num} SARIMAX...", end=" ", flush=True)
        t0 = time.time()
        try:
            order = (arima_fit.order if "ARIMA" in results_val else (1, d_ord, 1))
            sarimax_fit = SARIMAX(
                y_train,
                exog=sarimax_tr_exog,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)
 
            print(f"AICc={sarimax_fit.aicc:.1f} ({time.time()-t0:.0f}s)", flush=True)
            txt_lines.append(f"  SARIMAX: ordem{order}  AICc={sarimax_fit.aicc:.1f}")
 
            save_resid_plot(sarimax_fit.resid,
                            f"Resíduos SARIMAX V{val_num} | {uf_label}",
                            uf_out_dir / f"v{val_num}_sarimax_resid.png")
 
            sx_lo, sx_hi, sx_med = {}, {}, None
            for a in PI_LEVELS:
                fc_obj = sarimax_fit.get_forecast(steps=h_total, exog=sarimax_fc_exog)
                fc_df  = fc_obj.summary_frame(alpha=1-a)
                if sx_med is None: sx_med = fc_df["mean"].values
                sx_lo[a] = fc_df["mean_ci_lower"].values
                sx_hi[a] = fc_df["mean_ci_upper"].values
 
            sx_med_i = inv(sx_med)
            sx_lo_i  = {a: inv(sx_lo[a]) for a in PI_LEVELS}
            sx_hi_i  = {a: inv(sx_hi[a]) for a in PI_LEVELS}
            tgt_sx   = sx_med_i[gap:]
            m_sx = error_metrics(y_tgt_raw, tgt_sx, "SARIMAX", tgt_sx,
                                 {a: sx_lo_i[a][gap:] for a in PI_LEVELS},
                                 {a: sx_hi_i[a][gap:] for a in PI_LEVELS}, PI_LEVELS)
            results_val["SARIMAX"] = {"median":sx_med_i,"lower":sx_lo_i,
                                      "upper":sx_hi_i,"metrics":m_sx}
            txt_lines.append(f"    MAE={m_sx['MAE']:.1f}  RMSE={m_sx['RMSE']:.1f}  "
                             f"MAPE={m_sx['MAPE']:.1f}%  WIS={m_sx['WIS']:.2f}")
            save_forecast_plot(tr["date"].values, y_tr_raw,
                               tgt["date"].values, y_tgt_raw,
                               fc_dates, sx_med_i, sx_lo_i, sx_hi_i,
                               f"SARIMAX V{val_num} | {uf_label}",
                               uf_out_dir / f"v{val_num}_sarimax_forecast.png")
        except Exception as e:
            print(f"ERRO ({time.time()-t0:.0f}s)", flush=True)
            txt_lines.append(f"  [ERRO] SARIMAX: {e}")
 
        if not results_val:
            continue
 
        best_name = min(results_val,
                        key=lambda m: results_val[m]["metrics"].get("WIS", np.inf))
        txt_lines.append(f"  ★ Melhor modelo V{val_num}: {best_name} "
                         f"(WIS={results_val[best_name]['metrics'].get('WIS',np.nan):.2f})")
 
        # Gráfico comparativo
        fig_cmp, ax_cmp = plt.subplots(figsize=(14,4))
        ax_cmp.plot(tr["date"].values,  y_tr_raw,  color="#2166ac", lw=0.7, label="Treino")
        ax_cmp.plot(tgt["date"].values, y_tgt_raw, color="black",   lw=1.2, label="Observado")
        colors_cmp = {"ARIMA":"#e08214","ETS":"#542788","SARIMAX":"#27ae60"}
        for mn, mr in results_val.items():
            wis_v = mr["metrics"].get("WIS", np.nan)
            ax_cmp.plot(fc_dates, mr["median"], lw=1.0, ls="--",
                        color=colors_cmp.get(mn,"gray"),
                        label=f"{mn} WIS={wis_v:.2f}")
        ax_cmp.xaxis.set_major_locator(mdates.YearLocator())
        ax_cmp.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax_cmp.tick_params(axis="x", rotation=45, labelsize=7)
        ax_cmp.set_title(f"Comparação V{val_num} | {uf_label}",
                         fontsize=9, fontweight="bold")
        ax_cmp.legend(fontsize=7); ax_cmp.spines[["top","right"]].set_visible(False)
        plt.tight_layout()
        fig_cmp.savefig(uf_out_dir / f"v{val_num}_comparacao.png",
                        dpi=120, bbox_inches="tight")
        plt.close(fig_cmp)
 
        # Coleta resultados
        for mn, mr in results_val.items():
            all_metrics.append({
                "uf": uf_label, "uf_code": uf_code,
                "validation": val_num, "model": mn,
                "is_best": mn == best_name, **mr["metrics"],
            })
            for i, (d, obs) in enumerate(zip(tgt["date"].values, y_tgt_raw)):
                row = {"uf": uf_label, "uf_code": uf_code,
                       "validation": val_num, "model": mn,
                       "is_best": mn == best_name,
                       "date": d, "y_true": obs,
                       "median": mr["median"][gap+i]}
                for a in PI_LEVELS:
                    row[f"lower_{int(a*100)}"] = mr["lower"][a][gap+i]
                    row[f"upper_{int(a*100)}"] = mr["upper"][a][gap+i]
                all_forecasts.append(row)
 
print(f"\nPipeline UF: {(time.time()-t0_total)/60:.1f} min total")
 
# ===========================================================================
# III. Exportação
# ===========================================================================
print("Exportando parquets...")
df_metrics   = pd.DataFrame(all_metrics)
df_forecasts = pd.DataFrame(all_forecasts)
df_metrics.to_parquet(OUT_DIR / "metrics_classical.parquet",    index=False)
df_forecasts.to_parquet(OUT_DIR / "forecasts_classical.parquet", index=False)
 
with open(OUT_DIR / "relatorio_modelos.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(txt_lines))
print(f"  Relatório: {OUT_DIR / 'relatorio_modelos.txt'}")
 
# ===========================================================================
# IV. Gráficos por UF (já é o nível do loop — um plot por modelo/validação)
# ===========================================================================
print("Gerando mosaicos por UF...")
uf_dir = OUT_DIR / "plots_uf"
uf_dir.mkdir(exist_ok=True)
 
ufs_sorted = sorted(df_forecasts["uf"].unique())
N = len(ufs_sorted)
N_COLS = 6
N_ROWS = int(np.ceil(N / N_COLS))
 
for val_num in [1,2,3,4]:
    df_v = df_forecasts[df_forecasts["validation"] == val_num]
    if df_v.empty: continue
    for model_name in df_v["model"].unique():
        df_m = df_v[df_v["model"] == model_name]
        fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(24, N_ROWS*3.5),
                                 constrained_layout=True)
        axes_flat = axes.flatten()
        pal = ["#f4a582","#d6604d","#b2182b","#67001f"]
        for i, uf in enumerate(ufs_sorted):
            ax  = axes_flat[i]
            sub = df_m[df_m["uf"] == uf].sort_values("date")
            if sub.empty: ax.set_visible(False); continue
            ax.plot(sub["date"], sub["y_true"], color="black", lw=1.0, label="Observado")
            for j, a in enumerate(sorted(PI_LEVELS, reverse=True)):
                ax.fill_between(sub["date"],
                                sub[f"lower_{int(a*100)}"],
                                sub[f"upper_{int(a*100)}"],
                                alpha=0.2, color=pal[j])
            ax.plot(sub["date"], sub["median"], color="#d6604d", lw=1.0,
                    ls="--", label="Mediana")
            ax.set_title(uf, fontsize=8, fontweight="bold")
            ax.xaxis.set_major_locator(mdates.YearLocator(2))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax.tick_params(axis="x", labelsize=6, rotation=45)
            ax.tick_params(axis="y", labelsize=6)
            ax.spines[["top","right"]].set_visible(False)
        for j in range(N, len(axes_flat)):
            axes_flat[j].set_visible(False)
        fig.suptitle(f"Validation {val_num} — {model_name} | Casos por UF",
                     fontsize=11, fontweight="bold")
        fig.savefig(uf_dir / f"v{val_num}_{model_name}_uf_mosaic.png",
                    dpi=130, bbox_inches="tight")
        plt.close(fig)
 
# ===========================================================================
# V. Heatmap WIS modelo × UF por Validation
# ===========================================================================
print("Gerando heatmaps WIS...")
wis_uf = df_metrics.groupby(["validation","model","uf"])["WIS"].mean().reset_index()
 
for val_num in [1,2,3,4]:
    sub = wis_uf[wis_uf["validation"] == val_num]
    if sub.empty: continue
    pivot  = sub.pivot(index="model", columns="uf", values="WIS").round(2)
    models = pivot.index.tolist()
    states = pivot.columns.tolist()
 
    fig, ax = plt.subplots(figsize=(max(12, len(states)*0.75), max(4, len(models)*0.6)))
    im = ax.imshow(pivot.values, cmap="Reds",
                   vmin=np.nanpercentile(pivot.values, 5),
                   vmax=np.nanpercentile(pivot.values, 95),
                   aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="WIS")
    ax.set_xticks(range(len(states))); ax.set_yticks(range(len(models)))
    ax.set_xticklabels(states, fontsize=8)
    ax.set_yticklabels(models, fontsize=8)
    ax.set_xlabel("State", fontsize=9); ax.set_ylabel("Model", fontsize=9)
    vmax_ann = np.nanpercentile(pivot.values, 80)
    for i in range(len(models)):
        for j in range(len(states)):
            v = pivot.iloc[i,j]
            if np.isnan(v): continue
            ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                    fontsize=6.5, color="white" if v > vmax_ann else "black")
    ax.set_title(f"Validation test - {val_num} (WIS)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"heatmap_wis_v{val_num}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: heatmap_wis_v{val_num}.png")
 
# ===========================================================================
# VI. Resumo
# ===========================================================================
summary = (df_metrics.groupby(["validation","model"])[["MAE","RMSE","MAPE","WIS"]]
           .mean().round(2))
print("\nResumo médio (todas as UFs):")
print(summary.to_string())
print(f"\n✓ Concluído em {(time.time()-t0_total)/60:.1f} min. Resultados em: {OUT_DIR}")

[1/26] UF: AC
    V1 ARIMA... SARIMA(2, 0, 0)x(1, 0, 0, 52) (66s)
    V1 ETS... trend=None  seasonal=None  damped=False (0s)
    V1 SARIMAX... AICc=598.5 (1s)
    V2 ARIMA... SARIMA(2, 0, 0)x(1, 0, 0, 52) (107s)
    V2 ETS... trend=None  seasonal=None  damped=False (0s)
    V2 SARIMAX... AICc=654.7 (1s)
    V3 ARIMA... SARIMA(2, 0, 0)x(1, 0, 0, 52) (124s)
    V3 ETS... trend=None  seasonal=None  damped=False (0s)
    V3 SARIMAX... AICc=684.6 (2s)
    V4 ARIMA... SARIMA(2, 0, 0)x(1, 0, 0, 52) (124s)
    V4 ETS... trend=None  seasonal=None  damped=False (0s)
    V4 SARIMAX... AICc=710.5 (1s)
[2/26] UF: AL
    V1 ARIMA... SARIMA(2, 1, 0)x(1, 0, 0, 52) (68s)
    V1 ETS... trend=None  seasonal=None  damped=False (0s)
    V1 SARIMAX... AICc=253.0 (1s)
    V2 ARIMA... SARIMA(2, 1, 0)x(1, 0, 0, 52) (85s)
    V2 ETS... trend=None  seasonal=None  damped=False (0s)
    V2 SARIMAX... AICc=261.2 (1s)
    V3 ARIMA... SARIMA(2, 1, 0)x(1, 0, 0, 52) (98s)
    V3 ETS... trend=None  seasonal=None  damped

### 4. Rascunho